In [1]:
import os

import torch
import torch.nn as nn

import sys
sclembas = '/home/hmbaghda/Projects/scLEMBAS'
sys.path.insert(1, os.path.join(sclembas))
from scLEMBAS.model.model_utilities import update_with_defaults


import sys, os
cpa_path = '/home/hmbaghda/Projects/scLEMBAS/notebooks/cpa'
sys.path.insert(1, os.path.join(cpa_path))
from cpa import CPA
from cpa._data import AnnDataSplitter
from cpa._task import CPATrainingPlan
from scvi.dataloaders import DataSplitter
from tqdm import tqdm
from pytorch_lightning.callbacks import EarlyStopping
from scvi.train._callbacks import SaveBestState

[rank: 0] Global seed set to 0


In [2]:
n_cores = 12
os.environ["OMP_NUM_THREADS"] = str(n_cores)
os.environ["MKL_NUM_THREADS"] = str(n_cores)
os.environ["OPENBLAS_NUM_THREADS"] = str(n_cores)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(n_cores)
os.environ["NUMEXPR_NUM_THREADS"] = str(n_cores)

seed = 888

# device = "cuda" if torch.cuda.is_available() else "cpu"
device = 'cpu'
data_path = '/nobackup/users/hmbaghda/scLEMBAS/analysis'

In [3]:
# https://cpa-tools.readthedocs.io/en/latest/tutorials/combosciplex.html#Data-Loading

import scanpy as sc
adata = sc.read(os.path.join(data_path, 'raw', 'combo_sciplex_prep_hvg_filtered.h5ad'))
adata.X = adata.layers['counts'].copy()

In [4]:
import random
random.seed(seed)
adata.obs['cell_types_rand'] = random.choices(['macrophages', 'T_cells', 'B_cells'], k=adata.obs.shape[0])
adata.obs['cat_2'] = random.choices(['A', 'B', 'C', 'D'], k=adata.obs.shape[0])

In [5]:
# import random
# random.seed(seed)
# subset_barcodes = random.sample(population = adata.obs.index.tolist(), k = round(adata.shape[0]*0.01))

# adata = adata[subset_barcodes, :]

In [ ]:
CPA.setup_anndata(adata,
                      perturbation_key='condition_ID',
                      dosage_key='log_dose',
                      control_group='CHEMBL504',
                      batch_key=None,
                      is_count_data=True,
                      categorical_covariate_keys=['cell_types_rand', 'cat_2'],
                      deg_uns_key='rank_genes_groups_cov',
                      deg_uns_cat_key='cov_drug_dose',
                      max_comb_len=2,
                     )

100%|██████████████████████████████████████████| 32/32 [00:00<00:00, 902.56it/s]


In [ ]:
ae_hparams = {
    "n_latent": 128,
    "recon_loss": "gauss",
    "doser_type": "logsigm",
    "n_hidden_encoder": 512,
    "n_layers_encoder": 3,
    "n_hidden_decoder": 512,
    "n_layers_decoder": 1,
    "use_batch_norm_encoder": True,
    "use_layer_norm_encoder": False,
    "use_batch_norm_decoder": True,
    "use_layer_norm_decoder": False,
    "dropout_rate_encoder": 0.1,
    "dropout_rate_decoder": 0.1,
    "variational": False,
    "seed": 434,
}

trainer_params = {
    "n_epochs_kl_warmup": None,
    "n_epochs_pretrain_ae": 30,
    "n_epochs_adv_warmup": 50,
    "n_epochs_mixup_warmup": 3,
    "mixup_alpha": 0.1,
    "adv_steps": 2,
    "n_hidden_adv": 64,
    "n_layers_adv": 2,
    "use_batch_norm_adv": True,
    "use_layer_norm_adv": False,
    "dropout_rate_adv": 0.3,
    "reg_adv": 20.0,
    "pen_adv": 20.0,
    "lr": 0.0003,
    "wd": 4e-07,
    "adv_lr": 0.0003,
    "adv_wd": 4e-07,
    "adv_loss": "cce",
    "doser_lr": 0.0003,
    "doser_wd": 4e-07,
    "do_clip_grad": False,
    "gradient_clip_value": 1.0,
    "step_size_lr": 45,
}


self = CPA(adata=adata,
                split_key='split_1ct_MEC',
                train_split='train',
                valid_split='valid',
                test_split='ood',
                **ae_hparams,
               )